# 08 — WHEN-OTHERWISE vs CASE-WHEN

**DataFrame API `when/otherwise`** ↔ **Spark SQL `CASE WHEN`** — deriving & overwriting columns, NULL gotchas, and multi-branch age categorization.

In [0]:
# Setup: SparkSession, employees DataFrame (NULLs + duplicate row), temp view for Spark SQL
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, expr, count
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("conditional-constructs").getOrCreate()

emp_data = [
    (1, "Manish", 26, 20000, "INDIA", "IT"),
    (2, "Rahul", None, 40000, "GERMANY", "ENGINEERING"),
    (3, "Pawan", 12, 60000, "INDIA", "SALES"),
    (4, "Roshini", 44, None, "UK", "ENGINEERING"),
    (5, "Raushan", 35, 70000, "INDIA", "SALES"),
    (6, None, 29, 200000, "UK", "IT"),
    (7, "Adam", 37, 65000, "US", "IT"),
    (8, "Chris", 16, 40000, "US", "SALES"),
    (None, None, None, None, None, None),
    (7, "Adam", 37, 65000, "US", "IT"),
]

schema = """
    emp_id INT,
    emp_name STRING,
    age INT,
    salary DOUBLE,
    country STRING,
    department STRING
"""
emp_df = spark.createDataFrame(data=emp_data, schema=schema)
emp_df.createOrReplaceTempView("employees_df")
emp_df.printSchema()
emp_df.show()



In [0]:
# Derive is_adult column — first matching branch wins, NULL age falls to otherwise('N/A')
(
    emp_df
    .withColumn("is_adult", F.when(col("age") < 18, "No")
                            .when(col("age") >= 18, "Yes")
                            .otherwise("N/A"))
).show()

In [0]:
# Spark SQL Equivalent — CASE WHEN for is_adult (NULL age falls to ELSE 'N/A')
spark.sql("""
    SELECT *,
        CASE 
            WHEN age < 18 THEN 'No'
            WHEN age >= 18 THEN 'Yes'
            ELSE 'N/A'
        END AS is_adult
    FROM employees_df       
""").show()

In [0]:
# Overwrite the existing 'age' column — NULL-fill with 19, then derive is_adult

# Consider age = NULL as adult, with default value of 19 age
(
    emp_df
    .withColumn("age", F.when(col("age").isNull(), lit(19)).otherwise(col("age"))) # In this case if otherwise is missed all value will get set to NULL by default
    .withColumn("is_adult", F.when(col("age") < 18, "No").otherwise("Yes"))
).show()

In [0]:
# Variant 1 — CTE: NULL-fill age first, then derive is_adult
spark.sql("""
WITH cte AS (
    SELECT emp_id,
        CASE 
            WHEN age IS NULL THEN 19
            ELSE age
        END AS age,
        salary, 
        country,
        department
    FROM employees_df  
)

SELECT *, 
    CASE 
        WHEN age < 18 THEN "No"
        ELSE "Yes"
    END AS is_adult
FROM cte
""").show()

print("==============================================")

# Spark SQL Equivalent
spark.sql("""

    SELECT emp_id,
        CASE 
            WHEN age IS NULL THEN 19
            ELSE age
        END AS age,
        salary, 
        country,
        department,
        CASE
            WHEN age IS NULL OR age >= 18 THEN 'Yes'
            ELSE 'No'
        END AS is_adult
    FROM employees_df  

""").show()


In [0]:
# Categorize employees based on age, as follows
# Age 0-17  ==> Minor
# Age >=18  ==> Adult
# Age >=60  ==> Senior Citizen
# Otherwise ==> Invalid

(
    emp_df
    .withColumn(
        "age_category", 
            F.when( (col("age") >= 0) & (col("age") < 18), "Minor")
            .when((col("age") >= 18) & (col("age") < 60), "Adult")
            .when(col("age") >= 60, "Senior Citizen")
            .otherwise("Unknown")
    )
).show()

In [0]:
# Spark SQL Equivalent — CASE-WHEN for age_category (NULL ages fall to ELSE 'Unknown')
spark.sql("""
    SELECT *,
        CASE 
            WHEN age > 0 AND age < 18 THEN 'Minor'
            WHEN age >= 18 AND age < 60 THEN 'Adult'
            WHEN age >= 60 THEN 'Senior Citizen'
            ELSE 'Unknown'
        END AS age_category
    FROM employees_df
""").show()

# Quick Revision Notes (Handwritten-Style) — WHEN-OTHERWISE & CASE-WHEN

## 1 — Syntax map

* DF API: `F.when(cond, val).when(cond, val).otherwise(val)`
* SQL: `CASE WHEN c THEN v WHEN c THEN v ELSE v END`
* Both evaluate **top-down, first match wins** — remaining branches skipped

## 2 — NULL is the biggest gotcha ⚠️

* NULL in any comparison (`age < 18`, `age >= 18`) → result is NULL, NOT true
* So NULL age does NOT match any `when` → lands in `.otherwise()` / `ELSE`
* Cell 3: NULL age → `N/A` (caught by otherwise), not "No" or "Yes"
* 💡 Handle NULLs explicitly, BEFORE range checks:
  `F.when(col("age").isNull(), default).otherwise(col("age"))`

## 3 — Missing `.otherwise()` → unmatched rows become NULL

* Same as SQL `CASE` with no `ELSE` → NULL
* Cell 5 comment: drop the `.otherwise(col("age"))` → ALL ages become NULL

## 4 — Overwrite an existing column

* `withColumn("age", ...)` with the SAME name replaces it in place
* Null-fill pattern: `F.when(col("age").isNull(), lit(19)).otherwise(col("age"))`
* Two `withColumn` calls in a chain → 2nd sees the NEW age (19 → "Yes")

## 5 — Multi-branch categorization (Cell 7 / 8)

* Age 0–17 → Minor, 18–59 → Adult, ≥60 → Senior Citizen, else Unknown
* Use exclusive ranges: `(age >= 0) & (age < 18)`
* ⚠️ Cell 8 SQL uses `age > 0` (not `>= 0`) → age = 0 gives 'Unknown' there

## DF API vs SQL

| DataFrame API | Spark SQL |
| --- | --- |
| `F.when(c, v)` | `WHEN c THEN v` |
| chained `.when(...)` | more `WHEN` lines |
| `.otherwise(v)` | `ELSE v` |
| no `.otherwise()` → NULL | no `ELSE` → NULL |
| `&` / `\|` joins (wrap in `()`) | `AND` / `OR` |
| `col("x").isNull()` | `x IS NULL` |
| `col("x").isNotNull()` | `x IS NOT NULL` |

## Quick Recall Box ⭐

* WHEN = when, OTHERWISE = ELSE — same engine underneath
* First matching branch wins, rest skipped
* NULL never matches a comparison → handle it first or it slips to otherwise/ELSE
* No otherwise/ELSE → unmatched = NULL (dangerous default!)
* `&` needs parentheses: `(c1) & (c2)`, not `c1 & c2`
* `withColumn("same_name", ...)` overwrites; later links see updated value